<a href="https://colab.research.google.com/github/JyothiMekalaa/Chatbot/blob/main/Vector_Based_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install Required Libraries**

In [1]:
!pip install sentence-transformers faiss-cpu flask pyngrok openai python-dotenv nltk



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 19.9 MB/s eta 0:00:00


In [2]:
# connect to drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Import Libraries & Setup Constants**

In [3]:
import os
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize
import nltk
from tqdm import tqdm

nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

**SETTINGS**

In [4]:
EMB_MODEL = "all-MiniLM-L6-v2"
INDEX_PATH = "faiss_index.bin"
DOCS_META = "docs_meta.json"
DOCS_FOLDER = "/content/drive/MyDrive/docs"   # your .txt files folder
CHUNK_SIZE = 400
CHUNK_OVERLAP = 100


**Read Documents**

In [5]:
def read_documents(folder):
    docs = []
    for fname in os.listdir(folder):
        if fname.endswith(".txt"):
            path = os.path.join(folder, fname)
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
            docs.append({"id": fname, "text": text})
    return docs

docs = read_documents(DOCS_FOLDER)
print("Loaded documents:", len(docs))


Loaded documents: 1


**Chunk Text**

In [6]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    sents = sent_tokenize(text)
    chunks, cur = [], ""

    for s in sents:
        if len(cur) + len(s) <= chunk_size:
            cur += " " + s
        else:
            chunks.append(cur.strip())
            cur = s

    if cur.strip():
        chunks.append(cur.strip())

    return chunks


**Build FAISS Index**

In [7]:
def build_index():
    model = SentenceTransformer(EMB_MODEL)
    docs = read_documents(DOCS_FOLDER)

    all_chunks = []
    meta = []

    for d in docs:
        chunks = chunk_text(d['text'])
        for i, c in enumerate(chunks):
            meta.append({"doc_id": d['id'], "chunk_id": i, "text": c})
            all_chunks.append(c)

    print("Total chunks:", len(all_chunks))

    if not all_chunks:
        print("No chunks found.")
        return

    embeddings = model.encode(all_chunks, batch_size=64, show_progress_bar=True)
    embeddings = np.array(embeddings).astype("float32")

    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    faiss.write_index(index, INDEX_PATH)

    with open(DOCS_META, "w") as f:
        json.dump(meta, f, indent=2)

    print("Index built successfully.")


**Run Index Builder**

In [8]:
if __name__=="__main__":
  import nltk
  nltk.download('punkt_tab')
  build_index()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks: 57


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Index built successfully.


**Load Model & Index**

In [9]:
import openai
from dotenv import load_dotenv
load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")

embedder = SentenceTransformer(EMB_MODEL)
index = faiss.read_index(INDEX_PATH)

with open(DOCS_META, "r") as f:
    meta = json.load(f)

**Retrieval Functions**

In [10]:
def embed_query(q):
    q_emb = embedder.encode([q]).astype("float32")
    faiss.normalize_L2(q_emb)
    return q_emb

def retrieve(q, k=5):
    q_emb = embed_query(q)
    D, I = index.search(q_emb, k)
    results = [meta[i] for i in I[0]]
    return results

def build_context(chunks):
    ctx = ""
    for c in chunks:
        ctx += f"[{c['doc_id']} | chunk:{c['chunk_id']}]\n{c['text']}\n\n"
    return ctx


**PROMPT TEMPLATE**

In [11]:
PROMPT_TEMPLATE = """
You are a helpful assistant. Use ONLY the context below to answer.

Context:
{context}

User Question:
{question}

Final Answer (short and accurate):
"""


**Generate Answer Using OpenAI**

In [12]:
def generate_answer(question, retrieved):
    context = build_context(retrieved)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    resp = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role":"user","content":prompt}],
        max_tokens=300
    )
    return resp['choices'][0]['message']['content']


**FLASK API**

In [13]:

!pip install flask pyngrok

In [15]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "36SZq7LhwNflXok8USgvy1Dr1ne_3ZRfNzqdZVkwkUiUQopnf"   # <-- PUT REAL TOKEN HERE

# Apply authtoken
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [16]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
from threading import Thread
import time

# Dummy functions to avoid crash
def retrieve(q):
    return ["chunk_1", "chunk_2"]

def generate_answer(q, chunks):
    return f"Answer for: {q}"

# Create Flask app
app = Flask(__name__)

@app.route("/query", methods=["POST"])
def query():
    data = request.json
    q = data.get("query", "")

    if not q:
        return jsonify({"error": "Query missing"}), 400

    chunks = retrieve(q)
    answer = generate_answer(q, chunks)

    return jsonify({"answer": answer, "retrieved": chunks})

# Start Flask in a thread (so Colab does not block)
def start_flask():
    app.run(port=5000)

flask_thread = Thread(target=start_flask)
flask_thread.daemon = True
flask_thread.start()

# Wait for Flask to start
time.sleep(2)

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print("🔥 Your Public API URL:", public_url)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


🔥 Your Public API URL: NgrokTunnel: "https://gregorio-unascertained-arrogantly.ngrok-free.dev" -> "http://localhost:5000"


In [21]:
!curl -X POST https://gregorio-unascertained-arrogantly.ngrok-free.dev/query \
     -H "Content-Type: application/json" \
     -d '{"query":"Hello"}'

INFO:werkzeug:127.0.0.1 - - [08/Dec/2025 15:10:56] "POST /query HTTP/1.1" 200 -


{"answer":"Answer for: Hello","retrieved":["chunk_1","chunk_2"]}
